In [1]:
import pandas as pd
import numpy as np
from itertools import product
from collections import Counter
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
import seaborn as sns
import time
import warnings
import joblib      # ← model persistence
import json        # ← metadata saving
import os          # ← directory creation
warnings.filterwarnings('ignore')

try:
    from tqdm import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm', '-q'])
    from tqdm import tqdm
    TQDM_AVAILABLE = True
import random


# ============================================
# AMINO ACID PROPERTY TABLES  (module-level)
# ============================================
MOL_WEIGHTS = {
    'A': 89.1,  'R': 174.2, 'N': 132.1, 'D': 133.1, 'C': 121.2,
    'Q': 146.2, 'E': 147.1, 'G': 75.1,  'H': 155.2, 'I': 131.2,
    'L': 131.2, 'K': 146.2, 'M': 149.2, 'F': 165.2, 'P': 115.1,
    'S': 105.1, 'T': 119.1, 'W': 204.2, 'Y': 181.2, 'V': 117.1
}
HYDROPATHY = {
    'A': 1.8,  'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8,  'K': -3.9, 'M': 1.9,  'F': 2.8,  'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}
PKA_VALUES  = {'D': 3.9, 'E': 4.3, 'C': 8.3, 'Y': 10.1, 'H': 6.0, 'K': 10.5, 'R': 12.5}

ACIDIC      = set('DE')
BASIC       = set('RHK')
HYDROPHOBIC = set('AILMFWYV')
HYDROPHILIC = set('STNQDE')
AROMATIC    = set('FWY')
ALIPHATIC   = set('AVILM')
POLAR       = set('STNQ')
TINY        = set('AGST')
SMALL       = set('AGSTP')
CHARGED_POS = set('KRH')
CHARGED_NEG = set('DE')
NONPOLAR    = set('ACFILMPVWY')

AA_STD = 'ACDEFGHIKLMNPQRSTVWY'
G_GAP_RANGE = range(2, 11)   # g = 2 … 10

# ============================================================
# MODEL SAVING DIRECTORY
# ============================================================
MODEL_SAVE_DIR = 'saved_models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

def make_safe_name(s):
    """Convert augmentation/model name to a filesystem-safe string."""
    return s.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '-')

def save_model_artifacts(model, pca, scaler, feature_names, metrics, aug_name, model_name):
    """
    Save all artifacts needed to reproduce predictions for one
    (augmentation, model) combination.

    Directory layout:
        saved_models/
            <AugMethod>__<ModelName>/
                model.pkl           ← trained sklearn estimator
                pca.pkl             ← fitted PCA transformer
                scaler.pkl          ← fitted StandardScaler
                feature_names.json  ← ordered list of raw feature names
                metadata.json       ← R², RMSE, MAE, CV-R², dataset size, …

    Loading example (in any later script):
        import joblib, json
        run_dir = 'saved_models/Bootstrap__Gradient_Boosting'
        model   = joblib.load(f'{run_dir}/model.pkl')
        pca     = joblib.load(f'{run_dir}/pca.pkl')
        scaler  = joblib.load(f'{run_dir}/scaler.pkl')
        feat_names = json.load(open(f'{run_dir}/feature_names.json'))
        # Then: X_raw → feature_extraction → scaler.transform → pca.transform → model.predict
    """
    run_dir = os.path.join(
        MODEL_SAVE_DIR,
        f'{make_safe_name(aug_name)}__{make_safe_name(model_name)}'
    )
    os.makedirs(run_dir, exist_ok=True)

    joblib.dump(model,   os.path.join(run_dir, 'model.pkl'),  compress=3)
    joblib.dump(pca,     os.path.join(run_dir, 'pca.pkl'),    compress=3)
    joblib.dump(scaler,  os.path.join(run_dir, 'scaler.pkl'), compress=3)

    with open(os.path.join(run_dir, 'feature_names.json'), 'w') as f:
        json.dump(feature_names, f)

    with open(os.path.join(run_dir, 'metadata.json'), 'w') as f:
        json.dump({
            'augmentation' : aug_name,
            'model'        : model_name,
            'test_r2'      : round(metrics['r2'],   6),
            'rmse'         : round(metrics['rmse'],  6),
            'mae'          : round(metrics['mae'],   6),
            'cv_r2_mean'   : round(metrics['cv_mean'], 6),
            'cv_r2_std'    : round(metrics['cv_std'],  6),
            'dataset_size' : metrics['n'],
            'pca_components': metrics['n_pca'],
            'train_time_s' : round(metrics['train_time'], 4),
        }, f, indent=2)

    return run_dir


# ============================================
# EXTENDED FEATURE EXTRACTION (per-sequence)
# ============================================

def extract_advanced_features(sequence):
    seq = str(sequence).upper()
    length = len(seq)
    if length == 0:
        return {}

    features = {}

    # ── BASIC ──────────────────────────────────────────────────────────────
    features['length'] = length
    features['molecular_weight'] = sum(MOL_WEIGHTS.get(aa, 0) for aa in seq)

    # ── ESTIMATED pI ───────────────────────────────────────────────────────
    n_asp = seq.count('D'); n_glu = seq.count('E')
    n_cys = seq.count('C'); n_tyr = seq.count('Y')
    n_his = seq.count('H'); n_lys = seq.count('K'); n_arg = seq.count('R')
    features['estimated_pI'] = 7.0 + (n_his * 0.5 + n_lys + n_arg - n_asp - n_glu) * 0.5

    # ── NET CHARGE AT MULTIPLE pH ──────────────────────────────────────────
    for ph_val in [5.0, 6.0, 7.0, 7.4, 8.0, 9.0]:
        nc = 0
        if ph_val < 10.5: nc += n_lys
        if ph_val < 12.5: nc += n_arg
        if ph_val < 6.0:  nc += n_his
        if ph_val > 3.9:  nc -= n_asp
        if ph_val > 4.3:  nc -= n_glu
        if ph_val > 8.3:  nc -= n_cys
        if ph_val > 10.1: nc -= n_tyr
        features[f'net_charge_pH_{ph_val}']     = nc
        features[f'charge_density_pH_{ph_val}'] = nc / length

    # ── AROMATICITY / GRAVY / ALIPHATIC INDEX ──────────────────────────────
    aromatic_count = sum(1 for aa in seq if aa in AROMATIC)
    features['aromaticity']    = aromatic_count / length
    features['GRAVY']          = sum(HYDROPATHY.get(aa, 0) for aa in seq) / length
    features['aliphatic_index'] = 100 * (
        seq.count('A') / length +
        2.9 * seq.count('V') / length +
        3.9 * (seq.count('I') + seq.count('L')) / length
    )

    # ── pKa STATS ──────────────────────────────────────────────────────────
    pka_list = [PKA_VALUES[aa] for aa in seq if aa in PKA_VALUES]
    features['avg_pKa'] = np.mean(pka_list) if pka_list else 7.0
    features['pKa_std'] = np.std(pka_list)  if len(pka_list) > 1 else 0

    # ── AMINO ACID TYPE COUNTS & RATIOS (EXTENDED) ─────────────────────────
    counts = {
        'acidic'      : sum(1 for aa in seq if aa in ACIDIC),
        'basic'       : sum(1 for aa in seq if aa in BASIC),
        'hydrophobic' : sum(1 for aa in seq if aa in HYDROPHOBIC),
        'hydrophilic' : sum(1 for aa in seq if aa in HYDROPHILIC),
        'aromatic'    : aromatic_count,
        'aliphatic'   : sum(1 for aa in seq if aa in ALIPHATIC),
        'polar'       : sum(1 for aa in seq if aa in POLAR),
        'tiny'        : sum(1 for aa in seq if aa in TINY),
        'small'       : sum(1 for aa in seq if aa in SMALL),
        'charged_pos' : sum(1 for aa in seq if aa in CHARGED_POS),
        'charged_neg' : sum(1 for aa in seq if aa in CHARGED_NEG),
        'nonpolar'    : sum(1 for aa in seq if aa in NONPOLAR),
    }
    for cls, cnt in counts.items():
        features[f'num_{cls}']   = cnt
        features[f'ratio_{cls}'] = cnt / length

    # ── COMPOSITION RATIOS ─────────────────────────────────────────────────
    features['charge_to_polar_ratio']           = (counts['acidic'] + counts['basic']) / (counts['polar'] + 1)
    features['hydrophobic_to_hydrophilic_ratio']= counts['hydrophobic'] / (counts['hydrophilic'] + 1)
    features['aromatic_to_aliphatic_ratio']     = counts['aromatic']    / (counts['aliphatic']   + 1)
    features['pos_to_neg_charge_ratio']         = counts['charged_pos'] / (counts['charged_neg'] + 1)
    features['polar_to_nonpolar_ratio']         = counts['polar']       / (counts['nonpolar']    + 1)

    # ── INDIVIDUAL AA FREQUENCIES ──────────────────────────────────────────
    for aa in AA_STD:
        features[f'{aa}_freq'] = seq.count(aa) / length

    # ── DIPEPTIDE FREQUENCIES (full 400-dim, sparse stored) ───────────────
    if length > 1:
        di_counts = Counter(seq[i:i+2] for i in range(length - 1))
        total_di  = length - 1
        for dipep in (''.join(p) for p in product(AA_STD, repeat=2)):
            cnt = di_counts.get(dipep, 0)
            if cnt:
                features[f'dipep_{dipep}'] = cnt / total_di

    # ── TRIPEPTIDE FREQUENCIES ─────────────────────────────────────────────
    if length >= 3:
        tri_counts = Counter(seq[i:i+3] for i in range(length - 2))
        total_tri  = length - 2
        for tri, cnt in tri_counts.items():
            features[f'tripep_{tri}'] = cnt / total_tri

    # ── TETRAPEPTIDE FREQUENCIES ───────────────────────────────────────────
    if length >= 4:
        tetra_counts = Counter(seq[i:i+4] for i in range(length - 3))
        total_tetra  = length - 3
        for tetra, cnt in tetra_counts.items():
            features[f'tetrapep_{tetra}'] = cnt / total_tetra

    # ── PENTAPEPTIDE FREQUENCIES ───────────────────────────────────────────
    if length >= 5:
        penta_counts = Counter(seq[i:i+5] for i in range(length - 4))
        total_penta  = length - 4
        for penta, cnt in penta_counts.items():
            features[f'pentapep_{penta}'] = cnt / total_penta

    # ── G-GAP PEPTIDES  g = 2 … 10 ────────────────────────────────────────
    for g in G_GAP_RANGE:
        step = g + 1          # distance between the two flanking residues
        if length >= step + 1:
            gap_counts = Counter()
            for i in range(length - step):
                gap_counts[seq[i] + seq[i + step]] += 1
            total_gap = length - step
            for pat, cnt in gap_counts.items():
                features[f'gap{g}_{pat}'] = cnt / total_gap

    return features


# ─────────────────────────────────────────────────────────────────────────────
def extract_features_from_sequences(sequences):
    """Extract features for all sequences with a per-sequence progress bar."""
    print("  Extracting extended features (per sequence)...")
    t0 = time.time()
    seqs = list(sequences)
    all_dicts = []
    for seq in tqdm(seqs, desc='  Feature extraction', unit='seq',
                    ncols=80, colour='cyan'):
        all_dicts.append(extract_advanced_features(seq))
    df_feat = pd.DataFrame(all_dicts).fillna(0)
    print(f"  ✓ Done in {time.time()-t0:.1f}s  |  shape: {df_feat.shape}")
    return df_feat


# ============================================
# pH CLASS HELPER  (BALANCED CLASSES)
# ============================================

def classify_ph(ph):
    if ph < 6.5:   return "acidic"
    elif ph > 7.0: return "basic"
    else:          return "neutral"


def get_balanced_class_plan(y, n_synthetic):
    all_classes   = ["acidic", "neutral", "basic"]
    labels        = np.array([classify_ph(ph) for ph in y])
    class_indices = {cls: np.where(labels == cls)[0] for cls in all_classes}
    real_counts   = {cls: len(class_indices[cls])    for cls in all_classes}
    target        = max(real_counts.values()) + (n_synthetic // 3)
    class_n_synth = {cls: max(0, target - real_counts[cls]) for cls in all_classes}
    final_counts  = {cls: real_counts[cls] + class_n_synth[cls] for cls in all_classes}
    print(f"    Class plan  (target per class = {target}):")
    for cls in all_classes:
        print(f"      {cls:>8}: {real_counts[cls]:3d} real  "
              f"+ {class_n_synth[cls]:3d} synthetic  "
              f"= {final_counts[cls]:3d} total")
    return class_indices, class_n_synth


# ============================================
# BALANCED SYNTHETIC DATA GENERATION METHODS
# ============================================

def generate_bootstrap_samples(X, y, n_synthetic=100):
    print(f"\n  Generating balanced bootstrap samples...")
    class_indices, class_n_synth = get_balanced_class_plan(y, n_synthetic)
    sx_parts, sy_parts = [], []
    for cls, idx in class_indices.items():
        n = class_n_synth[cls]
        if n == 0 or len(idx) == 0: continue
        chosen = np.random.choice(idx, size=n, replace=True)
        sx_parts.append(X[chosen]); sy_parts.append(y[chosen])
    X_aug = np.vstack([X] + sx_parts); y_aug = np.hstack([y] + sy_parts)
    print(f"    ✓ Bootstrap: {len(X)} → {len(X_aug)} samples")
    return X_aug, y_aug


# ============================================
# SAFE COLORMAP HELPER
# ============================================

def get_colors(cmap_name, n):
    return mcm.get_cmap(cmap_name)(np.linspace(0, 1, max(n, 1)))


# ============================================
# MAIN PIPELINE
# ============================================
print("="*70)
print("COMPREHENSIVE AUGMENTATION & MODEL COMPARISON ANALYSIS")
print("  (Extended Features: dipeptides, tripeptides, tetrapeptides,")
print("   pentapeptides, g-gap g=2..10, extended AA class counts)")
print("="*70)

df = pd.read_excel('output_with_sequences.xlsx')
sequence_col = 'sequence'
target_col   = 'ph_optimum'

print(f"\nOriginal dataset: {len(df)} samples")
print(f"pH range: {df[target_col].min():.2f} - {df[target_col].max():.2f}")

orig_labels = df[target_col].apply(classify_ph)
print("\nOriginal class distribution:")
print(orig_labels.value_counts().to_string())

print("\nExtracting extended features...")
X_df              = extract_features_from_sequences(df[sequence_col])
feature_names_all = X_df.columns.tolist()
X_orig            = X_df.values
y_orig            = df[target_col].values

scaler    = StandardScaler()
X_orig_sc = scaler.fit_transform(X_orig)

# ============================================
# GENERATE SYNTHETIC DATA
# ============================================
print("\n" + "="*70)
print("GENERATING SYNTHETIC DATA")
print("="*70)

n_synthetic = 150
datasets = {}
datasets['Original']      = (X_orig_sc, y_orig)
datasets['Bootstrap']     = generate_bootstrap_samples(X_orig_sc, y_orig, n_synthetic)


print("\n--- Final class counts after augmentation ---")
for name, (_, y_d) in datasets.items():
    counts = Counter(classify_ph(v) for v in y_d)
    print(f"  {name:15s}: acidic={counts.get('acidic',0):3d}  "
          f"neutral={counts.get('neutral',0):3d}  "
          f"basic={counts.get('basic',0):3d}  (total={len(y_d)})")

print("\n✓ All augmentation methods completed!")

# ============================================
# MODELS
# ============================================
print("\n" + "="*70)
print("TRAINING & EVALUATING ALL COMBINATIONS")
print("="*70)

models = {
    'Linear Regression' : LinearRegression(),
    'Ridge Regression'  : Ridge(alpha=1.0),
    'Lasso Regression'  : Lasso(alpha=0.01, max_iter=5000),
    'SVR (Linear)'      : SVR(kernel='linear', C=1.0, cache_size=500),
    'SVR (RBF)'         : SVR(kernel='rbf',    C=1.0, gamma='scale', cache_size=500),
    'SVR (Polynomial)'  : SVR(kernel='poly',   degree=5, C=1.0, cache_size=500),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, max_depth=10,
                                                random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, max_depth=7,
                                                    random_state=42),
}

results_comparison = []
best_rf_info       = None
saved_model_log    = []          # ← track all saved paths
total_exp          = len(datasets) * len(models)

# Outer progress bar: augmentation methods
aug_pbar = tqdm(datasets.items(), desc='Augmentation methods',
                total=len(datasets), ncols=80, colour='green')

for method_name, (X_data, y_data) in aug_pbar:
    aug_pbar.set_description(f'Aug: {method_name:<15}')

    X_tr, X_te, y_tr, y_te = train_test_split(X_data, y_data,
                                               test_size=0.2, random_state=42)
    pca      = PCA(n_components=0.99, random_state=42)
    X_tr_pca = pca.fit_transform(X_tr)
    X_te_pca = pca.transform(X_te)

    tqdm.write(f"\n{'─'*70}")
    tqdm.write(f"  Augmentation: {method_name}  |  n={len(X_data)}  "
               f"|  PCA components: {X_tr_pca.shape[1]}")
    tqdm.write(f"{'─'*70}")

    # Inner progress bar: models
    model_pbar = tqdm(models.items(), desc='  Models', total=len(models),
                      ncols=80, colour='yellow', leave=False)

    for model_name, model in model_pbar:
        model_pbar.set_description(f'  {model_name:<22}')
        try:
            t0 = time.time()
            model.fit(X_tr_pca, y_tr)
            train_time = time.time() - t0

            y_pred = model.predict(X_te_pca)
            r2     = r2_score(y_te, y_pred)
            rmse   = np.sqrt(mean_squared_error(y_te, y_pred))
            mae    = mean_absolute_error(y_te, y_pred)

            n_folds = min(5, len(X_tr) // 5)
            if n_folds >= 2:
                cv      = cross_val_score(model, X_tr_pca, y_tr,
                                          cv=n_folds, scoring='r2', n_jobs=-1)
                cv_mean, cv_std = cv.mean(), cv.std()
            else:
                cv_mean, cv_std = r2, 0.0

            results_comparison.append({
                'Augmentation'  : method_name,
                'Model'         : model_name,
                'Dataset Size'  : len(X_data),
                'PCA Components': X_tr_pca.shape[1],
                'Test R2'       : r2,
                'RMSE'          : rmse,
                'MAE'           : mae,
                'CV R2'         : cv_mean,
                'CV Std'        : cv_std,
                'Train Time (s)': train_time,
            })
            tqdm.write(f"    {model_name:22s}  R²={r2:+.4f}  "
                       f"RMSE={rmse:.4f}  t={train_time:.2f}s")

            # ── SAVE MODEL ARTIFACTS ──────────────────────────────────────
            metrics = dict(r2=r2, rmse=rmse, mae=mae,
                           cv_mean=cv_mean, cv_std=cv_std,
                           n=len(X_data), n_pca=X_tr_pca.shape[1],
                           train_time=train_time)
            run_dir = save_model_artifacts(
                model, pca, scaler, feature_names_all,
                metrics, method_name, model_name
            )
            saved_model_log.append((method_name, model_name, run_dir, r2))
            tqdm.write(f"      💾 Saved → {run_dir}")
            # ─────────────────────────────────────────────────────────────

            if model_name in ('Random Forest', 'Gradient Boosting'):
                if best_rf_info is None or r2 > best_rf_info['r2']:
                    best_rf_info = dict(r2=r2, model=model, pca=pca,
                                        model_name=model_name,
                                        method_name=method_name)

        except Exception as e:
            tqdm.write(f"    {model_name:22s}  ERROR: {e}")

    model_pbar.close()

aug_pbar.close()

# ============================================
# PRINT SAVED MODEL SUMMARY
# ============================================
print("\n" + "="*70)
print("SAVED MODELS SUMMARY")
print("="*70)
saved_model_log.sort(key=lambda x: -x[3])   # sort by R² descending
for aug, mdl, path, r2 in saved_model_log:
    print(f"  R²={r2:+.4f}  {aug:12s} + {mdl:22s}  →  {path}/")

print(f"\n  All models saved under: ./{MODEL_SAVE_DIR}/")
print("""
  To load any model later:
      import joblib, json
      run_dir = 'saved_models/Bootstrap__Gradient_Boosting'
      model  = joblib.load(f'{run_dir}/model.pkl')
      pca    = joblib.load(f'{run_dir}/pca.pkl')
      scaler = joblib.load(f'{run_dir}/scaler.pkl')
      feat_names = json.load(open(f'{run_dir}/feature_names.json'))
      meta   = json.load(open(f'{run_dir}/metadata.json'))
      # Predict: scaler → pca → model
      X_sc   = scaler.transform(X_raw)
      X_pca  = pca.transform(X_sc)
      y_pred = model.predict(X_pca)
""")

# ============================================
# RESULTS SUMMARY
# ============================================
print("\n" + "="*70)
print("COMPLETE RESULTS")
print("="*70)

results_df = pd.DataFrame(results_comparison).sort_values('Test R2', ascending=False)
print("\nTop 20 Configurations:")
print(results_df.head(20).to_string(index=False))

best = results_df.iloc[0]
print(f"\n{'='*70}")
print(f"🏆 BEST CONFIGURATION")
print(f"{'='*70}")
print(f"  Augmentation : {best['Augmentation']}")
print(f"  Model        : {best['Model']}")
print(f"  Dataset Size : {int(best['Dataset Size'])}")
print(f"  Test R²      : {best['Test R2']:.4f}")
print(f"  RMSE         : {best['RMSE']:.4f}")
print(f"  MAE          : {best['MAE']:.4f}")
print(f"  CV R²        : {best['CV R2']:.4f} ± {best['CV Std']:.4f}")
print(f"  Train Time   : {best['Train Time (s)']:.2f}s")

# ============================================
# VISUALIZATION 1 – COMPREHENSIVE ANALYSIS
# ============================================
print("\n" + "="*70)
print("GENERATING VISUALIZATIONS")
print("="*70)

fig = plt.figure(figsize=(20, 12))
gs  = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
best_per_method = results_df.groupby('Augmentation')['Test R2'].max().sort_values(ascending=False)
colors = get_colors('viridis', len(best_per_method))
ax1.barh(range(len(best_per_method)), best_per_method.values, color=colors)
ax1.set_yticks(range(len(best_per_method)))
ax1.set_yticklabels(best_per_method.index, fontsize=9)
ax1.set_xlabel('Best Test R²', fontsize=10)
ax1.set_title('Best Performance by Augmentation Method', fontsize=11, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')
for i, v in enumerate(best_per_method.values):
    ax1.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)

ax2 = fig.add_subplot(gs[0, 1])
for mname in results_df['Model'].unique():
    md = results_df[results_df['Model'] == mname]
    ax2.scatter(md['Dataset Size'], md['Test R2'], label=mname, s=80, alpha=0.6)
ax2.set_xlabel('Dataset Size', fontsize=10)
ax2.set_ylabel('Test R²', fontsize=10)
ax2.set_title('Dataset Size vs Performance', fontsize=11, fontweight='bold')
ax2.legend(fontsize=7, loc='best', ncol=2)
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
pivot = results_df.pivot_table(values='Test R2', index='Model',
                               columns='Augmentation', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax3,
            cbar_kws={'label': 'R²'}, vmin=0, vmax=1, linewidths=0.5)
ax3.set_title('R² Heatmap: Method vs Model', fontsize=11, fontweight='bold')
plt.setp(ax3.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(ax3.get_yticklabels(), rotation=0,  fontsize=8)

ax4 = fig.add_subplot(gs[1, 0])
best_rmse = results_df.groupby('Augmentation')['RMSE'].min().sort_values()
ax4.barh(range(len(best_rmse)), best_rmse.values,
         color=get_colors('RdYlGn_r', len(best_rmse)))
ax4.set_yticks(range(len(best_rmse)))
ax4.set_yticklabels(best_rmse.index, fontsize=9)
ax4.set_xlabel('Best RMSE (lower = better)', fontsize=10)
ax4.set_title('RMSE', fontsize=11, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='x')
for i, v in enumerate(best_rmse.values):
    ax4.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=8)

ax5 = fig.add_subplot(gs[1, 1])
best_cv = (results_df.sort_values('CV R2', ascending=False)
                      .groupby('Augmentation', as_index=False)
                      .first()[['Augmentation', 'CV R2', 'CV Std']])
ax5.barh(range(len(best_cv)), best_cv['CV R2'], xerr=best_cv['CV Std'],
         color=get_colors('viridis', len(best_cv)), capsize=5, alpha=0.7)
ax5.set_yticks(range(len(best_cv)))
ax5.set_yticklabels(best_cv['Augmentation'], fontsize=9)
ax5.set_xlabel('CV R²', fontsize=10)
ax5.set_title('Cross-Validation Performance', fontsize=11, fontweight='bold')
ax5.grid(True, alpha=0.3, axis='x')

ax6 = fig.add_subplot(gs[1, 2])
model_perf = (results_df.groupby('Model')['Test R2']
              .agg(['mean', 'std']).sort_values('mean', ascending=False))
ax6.barh(range(len(model_perf)), model_perf['mean'], xerr=model_perf['std'],
         color=get_colors('Set2', len(model_perf)), capsize=5, alpha=0.7)
ax6.set_yticks(range(len(model_perf)))
ax6.set_yticklabels(model_perf.index, fontsize=9)
ax6.set_xlabel('Average Test R²', fontsize=10)
ax6.set_title('Model Performance (Averaged)', fontsize=11, fontweight='bold')
ax6.grid(True, alpha=0.3, axis='x')

ax7 = fig.add_subplot(gs[2, 0])
avg_time = results_df.groupby('Model')['Train Time (s)'].mean().sort_values()
ax7.barh(range(len(avg_time)), avg_time.values,
         color=get_colors('plasma', len(avg_time)))
ax7.set_yticks(range(len(avg_time)))
ax7.set_yticklabels(avg_time.index, fontsize=9)
ax7.set_xlabel('Avg Training Time (s)', fontsize=10)
ax7.set_title('Training Time by Model', fontsize=11, fontweight='bold')
ax7.grid(True, alpha=0.3, axis='x')
for i, v in enumerate(avg_time.values):
    ax7.text(v + 0.005, i, f'{v:.2f}s', va='center', fontsize=8)

ax8 = fig.add_subplot(gs[2, 1])
model_codes = results_df['Model'].astype('category').cat.codes
ax8.scatter(results_df['Train Time (s)'], results_df['Test R2'],
            c=model_codes, cmap='tab10', s=100, alpha=0.6)
ax8.set_xlabel('Training Time (s)', fontsize=10)
ax8.set_ylabel('Test R²', fontsize=10)
ax8.set_title('Performance vs Training Time', fontsize=11, fontweight='bold')
ax8.grid(True, alpha=0.3)
unique_models = results_df['Model'].unique()
handles = [plt.Line2D([0],[0], marker='o', color='w',
           markerfacecolor=mcm.get_cmap('tab10')(i / len(unique_models)),
           markersize=8, label=nm) for i, nm in enumerate(unique_models)]
ax8.legend(handles=handles, fontsize=7, loc='best', ncol=2)

ax9 = fig.add_subplot(gs[2, 2])
top10 = results_df.head(10).copy()
top10['Config'] = top10['Augmentation'].str[:8] + '\n' + top10['Model'].str[:12]
ax9.barh(range(len(top10)), top10['Test R2'].values,
         color=get_colors('RdYlGn', len(top10)))
ax9.set_yticks(range(len(top10)))
ax9.set_yticklabels(top10['Config'], fontsize=7)
ax9.set_xlabel('Test R²', fontsize=10)
ax9.set_title('Top 10 Configurations', fontsize=11, fontweight='bold')
ax9.grid(True, alpha=0.3, axis='x')
ax9.invert_yaxis()
for i, v in enumerate(top10['Test R2'].values):
    ax9.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=8)

plt.suptitle('Comprehensive Augmentation & Model Analysis\n(Extended Feature Set)',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('comprehensive_analysis.png', dpi=200, bbox_inches='tight')
plt.close()
print("✓ Saved comprehensive_analysis.png")

# ============================================
# VISUALIZATION 2 – PER-MODEL BREAKDOWN
# ============================================
fig2, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, mname in enumerate(results_df['Model'].unique()):
    ax  = axes.flatten()[idx]
    md  = results_df[results_df['Model'] == mname].sort_values('Test R2', ascending=False)
    ax.bar(range(len(md)), md['Test R2'].values,
           color=get_colors('viridis', len(md)))
    ax.set_xticks(range(len(md)))
    ax.set_xticklabels(md['Augmentation'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Test R²', fontsize=10)
    ax.set_title(mname, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1])
    for i, v in enumerate(md['Test R2'].values):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=7)
plt.tight_layout()
plt.savefig('model_specific_analysis.png', dpi=200, bbox_inches='tight')
plt.close()
print("✓ Saved model_specific_analysis.png")

# ============================================
# VISUALIZATION 3 – FEATURE IMPORTANCE
# ============================================
print("\n" + "="*70)
print("GENERATING FEATURE IMPORTANCE ANALYSIS")
print("="*70)

if best_rf_info is not None:
    best_model      = best_rf_info['model']
    best_pca        = best_rf_info['pca']
    best_model_name = best_rf_info['model_name']
    best_meth_name  = best_rf_info['method_name']
    print(f"  Best tree model : {best_model_name} ({best_meth_name})  "
          f"R²={best_rf_info['r2']:.4f}")

    pca_imp    = best_model.feature_importances_          # shape: (n_components,)
    components = best_pca.components_                     # shape: (n_components, n_pca_cols)
    orig_imp   = np.abs(components).T @ pca_imp           # shape: (n_pca_cols,)
    orig_imp  /= orig_imp.sum()

    n_pca_cols = components.shape[1]
    n_feat     = min(len(feature_names_all), n_pca_cols)

    print(f"  PCA input cols : {n_pca_cols}")
    print(f"  Total features : {len(feature_names_all)}")
    print(f"  Matched cols   : {n_feat}  (used for importance mapping)")

    feat_imp_df = pd.DataFrame({
        'Feature'   : feature_names_all[:n_feat],
        'Importance': orig_imp[:n_feat]
    }).sort_values('Importance', ascending=False)

    feat_imp_df.to_csv('feature_importances.csv', index=False)
    print("  ✓ Saved feature_importances.csv")

    fig3 = plt.figure(figsize=(22, 18))
    gs3  = fig3.add_gridspec(2, 2, hspace=0.42, wspace=0.42)

    # Panel A – Top 40 features
    ax_a   = fig3.add_subplot(gs3[0, :])
    top_n  = min(40, len(feat_imp_df))
    top_f  = feat_imp_df.head(top_n)
    ax_a.barh(range(top_n), top_f['Importance'].values,
              color=get_colors('viridis', top_n))
    ax_a.set_yticks(range(top_n))
    ax_a.set_yticklabels(top_f['Feature'].values, fontsize=8)
    ax_a.invert_yaxis()
    ax_a.set_xlabel('Relative Importance (back-projected through PCA)', fontsize=11)
    ax_a.set_title(
        f'Top {top_n} Most Important Features\n'
        f'(Model: {best_model_name}  |  Aug: {best_meth_name}  '
        f'|  R²={best_rf_info["r2"]:.4f})',
        fontsize=13, fontweight='bold'
    )
    ax_a.grid(True, alpha=0.3, axis='x')
    for i, v in enumerate(top_f['Importance'].values):
        ax_a.text(v + 0.0002, i, f'{v:.4f}', va='center', fontsize=7)

    # Panel B – Cumulative importance
    ax_b    = fig3.add_subplot(gs3[1, 0])
    cum_imp = feat_imp_df['Importance'].values.cumsum()
    ax_b.plot(range(1, len(cum_imp) + 1), cum_imp * 100, color='steelblue', lw=2)
    for thresh in [50, 80, 95]:
        idx_t = np.searchsorted(cum_imp, thresh / 100)
        ax_b.axhline(thresh, color='red',    lw=0.8, linestyle='--', alpha=0.6)
        ax_b.axvline(idx_t,  color='orange', lw=0.8, linestyle='--', alpha=0.6)
        ax_b.text(idx_t + 1, thresh + 1, f'{thresh}% @ {idx_t} feats',
                  fontsize=8, color='darkred')
    ax_b.set_xlabel('Number of Features (ranked)', fontsize=10)
    ax_b.set_ylabel('Cumulative Importance (%)', fontsize=10)
    ax_b.set_title('Cumulative Feature Importance Curve', fontsize=12, fontweight='bold')
    ax_b.grid(True, alpha=0.3)
    ax_b.set_ylim([0, 102])

    # Panel C – Importance by category
    ax_c = fig3.add_subplot(gs3[1, 1])

    def categorize_feature(fname):
        if fname.startswith('pentapep_'): return 'Pentapeptide'
        if fname.startswith('tetrapep_'): return 'Tetrapeptide'
        if fname.startswith('tripep_'):   return 'Tripeptide'
        if fname.startswith('dipep_'):    return 'Dipeptide'
        for g in G_GAP_RANGE:
            if fname.startswith(f'gap{g}_'): return f'G-gap (g={g})'
        if fname.endswith('_freq') and len(fname) == 6: return 'AA Frequency'
        if 'charge' in fname or 'pH' in fname: return 'Charge / pH'
        if fname in ('GRAVY', 'aromaticity', 'aliphatic_index',
                     'ratio_hydrophobic', 'ratio_hydrophilic',
                     'num_hydrophobic', 'num_hydrophilic',
                     'hydrophobic_to_hydrophilic_ratio'): return 'Hydrophobicity'
        if 'pKa' in fname or 'pI' in fname: return 'pKa / pI'
        if fname in ('length', 'molecular_weight'): return 'Basic'
        if 'ratio' in fname or 'num_' in fname: return 'Composition'
        return 'Other'

    feat_imp_df['Category'] = feat_imp_df['Feature'].apply(categorize_feature)
    cat_imp    = feat_imp_df.groupby('Category')['Importance'].sum().sort_values(ascending=False)
    cat_colors = get_colors('tab20', len(cat_imp))
    wedges, texts, autotexts = ax_c.pie(
        cat_imp.values, labels=cat_imp.index, autopct='%1.1f%%',
        colors=cat_colors, startangle=140, pctdistance=0.82, labeldistance=1.1
    )
    for t in texts:     t.set_fontsize(9)
    for t in autotexts: t.set_fontsize(8)
    ax_c.set_title('Feature Importance by Category', fontsize=12, fontweight='bold')

    plt.suptitle('Feature Importance Analysis (Extended Feature Set)',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.savefig('feature_importance_analysis.png', dpi=200, bbox_inches='tight')
    plt.close()
    print("  ✓ Saved feature_importance_analysis.png")

    print("\n  Top-5 features per category:")
    for cat, grp in feat_imp_df.groupby('Category'):
        top5 = grp.nlargest(5, 'Importance')
        print(f"\n    [{cat}]")
        for _, row in top5.iterrows():
            print(f"      {row['Feature']:<45s}  {row['Importance']:.5f}")
else:
    print("  ⚠ No tree-based model results found; skipping feature importance plot.")

# ============================================
# SAVE CSV + SUMMARY
# ============================================
results_df.to_csv('complete_augmentation_analysis.csv', index=False)
print("\n✓ Saved complete_augmentation_analysis.csv")

print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)
print("\n📊 Performance by Augmentation Method:")
print(results_df.groupby('Augmentation').agg(
    R2_mean=('Test R2','mean'), R2_max=('Test R2','max'),
    RMSE_min=('RMSE','min'), Train_s=('Train Time (s)','mean')
).round(4).to_string())

print("\n📊 Performance by Model:")
print(results_df.groupby('Model').agg(
    R2_mean=('Test R2','mean'), R2_max=('Test R2','max'),
    RMSE_min=('RMSE','min'), Train_s=('Train Time (s)','mean')
).round(4).to_string())

print("\n" + "="*70)
print("IMPROVEMENT ANALYSIS")
print("="*70)
orig_best = results_df[results_df['Augmentation'] == 'Original']['Test R2'].max()
aug_best  = results_df[results_df['Augmentation'] != 'Original']['Test R2'].max()
imp = (aug_best - orig_best) / abs(orig_best) * 100 if orig_best != 0 else 0
print(f"\n  Original best R²  : {orig_best:.4f}")
print(f"  Augmented best R² : {aug_best:.4f}")
print(f"  Improvement       : {imp:+.1f}%")

print(f"\n🎯 Recommended method : {best['Augmentation']}")
print(f"🎯 Recommended model  : {best['Model']}")

print("\n" + "="*70)
print("TOP 3 RECOMMENDATIONS")
print("="*70)
for rank, (_, row) in enumerate(results_df.head(3).iterrows(), 1):
    print(f"\n  {rank}. {row['Augmentation']} + {row['Model']}")
    print(f"     R²={row['Test R2']:.4f}  RMSE={row['RMSE']:.4f}  "
          f"Time={row['Train Time (s)']:.2f}s")

print("\n" + "="*70)
print("🎉 ANALYSIS COMPLETE!")
print("="*70)
print(f"  Methods tested     : {len(datasets)}")
print(f"  Models tested      : {len(models)}")
print(f"  Total experiments  : {len(results_df)}")
print(f"  G-gap range        : g = {min(G_GAP_RANGE)} … {max(G_GAP_RANGE)}")
print(f"  Output files:")
print(f"    comprehensive_analysis.png")
print(f"    model_specific_analysis.png")
print(f"    feature_importance_analysis.png")
print(f"    complete_augmentation_analysis.csv")
print(f"    feature_importances.csv")
print(f"    saved_models/  ({len(saved_model_log)} model directories)")


COMPREHENSIVE AUGMENTATION & MODEL COMPARISON ANALYSIS
  (Extended Features: dipeptides, tripeptides, tetrapeptides,
   pentapeptides, g-gap g=2..10, extended AA class counts)

Original dataset: 53 samples
pH range: 5.50 - 9.50

Original class distribution:
ph_optimum
basic      30
neutral    19
acidic      4

Extracting extended features...
  Extracting extended features (per sequence)...


  Feature extraction: 100%|███████████████████| 53/53 [00:00<00:00, 242.89seq/s]


  ✓ Done in 0.7s  |  shape: (53, 25948)

GENERATING SYNTHETIC DATA

  Generating balanced bootstrap samples...
    Class plan  (target per class = 80):
        acidic:   4 real  +  76 synthetic  =  80 total
       neutral:  19 real  +  61 synthetic  =  80 total
         basic:  30 real  +  50 synthetic  =  80 total
    ✓ Bootstrap: 53 → 240 samples

--- Final class counts after augmentation ---
  Original       : acidic=  4  neutral= 19  basic= 30  (total=53)
  Bootstrap      : acidic= 80  neutral= 80  basic= 80  (total=240)

✓ All augmentation methods completed!

TRAINING & EVALUATING ALL COMBINATIONS


Aug: Original       :   0%|                               | 0/2 [00:00<?, ?it/s]


──────────────────────────────────────────────────────────────────────
  Augmentation: Original  |  n=53  |  PCA components: 35
──────────────────────────────────────────────────────────────────────


...

    Random Forest           R²=+0.7432  RMSE=0.4949  t=0.16s


                                                                                
  Gradient Boosting     :  88%|████████████████▋  | 7/8 [00:02<00:00,  2.41it/s]

      💾 Saved → saved_models\Bootstrap__Random_Forest


                                                                                
Aug: Bootstrap      :  50%|███████████▌           | 1/2 [00:18<00:15, 15.12s/it]
                                                                                2.41it/s]
Aug: Bootstrap      : 100%|███████████████████████| 2/2 [00:19<00:00,  9.56s/it]


    Gradient Boosting       R²=+0.7446  RMSE=0.4935  t=0.22s
      💾 Saved → saved_models\Bootstrap__Gradient_Boosting

SAVED MODELS SUMMARY
  R²=+0.7446  Bootstrap    + Gradient Boosting       →  saved_models\Bootstrap__Gradient_Boosting/
  R²=+0.7432  Bootstrap    + Random Forest           →  saved_models\Bootstrap__Random_Forest/
  R²=+0.7426  Bootstrap    + Ridge Regression        →  saved_models\Bootstrap__Ridge_Regression/
  R²=+0.7426  Bootstrap    + Linear Regression       →  saved_models\Bootstrap__Linear_Regression/
  R²=+0.7419  Bootstrap    + Lasso Regression        →  saved_models\Bootstrap__Lasso_Regression/
  R²=+0.7068  Bootstrap    + SVR (Linear)            →  saved_models\Bootstrap__SVR_Linear/
  R²=+0.6924  Bootstrap    + SVR (RBF)               →  saved_models\Bootstrap__SVR_RBF/
  R²=+0.5340  Bootstrap    + SVR (Polynomial)        →  saved_models\Bootstrap__SVR_Polynomial/
  R²=-0.1311  Original     + SVR (Polynomial)        →  saved_models\Original__SVR_Polynomial